# धडा 03 - एजंटिक डिझाइन पॅटर्न्स

या धड्यात, प्रभावी एआय एजंट तयार करण्यासाठी तीन मूलभूत डिझाइन पॅटर्न्सचा अभ्यास करूया:

1. **सुस्पष्ट एजंट सूचनां** — एजंटच्या वर्तनाचे मार्गदर्शन करणाऱ्या अचूक, भूमिका-निर्धारित प्रम्प्ट्स तयार करणे  
2. **पायडँटिक मॉडेल्ससह रचना केलेले आउटपुट** — एजंटने प्रत्याशित, प्रमाणित डेटा परत करण्याची खात्री करणे  
3. **एकाकी जबाबदारी एजंट्स** — प्रत्येक एजंटला एक कार्य नीट पार पाडण्यास केंद्रित करणे  

आम्ही प्रत्येक पॅटर्न ट्रॅव्हल डेस्टिनेशन शिफारस करणाऱ्या परिस्थितीवर लागू करू, क्रमिकपणे एक असा प्रणाली तयार केली जाईल जी स्थळे सुचवू शकते, उपलब्धता तपासू शकते, आणि लॉजिस्टिक्स हाताळू शकते.


## सेटअप


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## नमुना 1: स्पष्ट एजंट सूचना

सर्वात प्रभावी नमुना सर्वात सोपा देखील आहे: आपल्या एजंटसाठी स्पष्ट, तपशीलवार सूचना लिहिणे.

चांगल्या सूचनांमध्ये ठरवलेले असते:
- **कोण** एजंट आहे (व्यक्तिमत्व आणि शैली)
- **काय** त्याने करायचे आहे (टप्प्याटपपी जबाबदाऱ्या)
- **कसे** त्याने वागायचे आहे (बाधा आणि शैली)

खाली, आम्ही एक ट्रॅव्हल कन्सीअर्ज एजंट तयार करतो ज्याला स्पष्ट सूचना दिल्या आहेत ज्यामुळे तो तयार करीत असलेल्या प्रत्येक उत्तराला आकार मिळतो.


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## पॅटर्न 2: पायडॅंटिक मॉडेल्ससह संरचित आउटपुट

मुक्त-संवाद मजकूर संभाषणासाठी उपयुक्त आहे, पण खालील प्रणालींना संरचित डेटाची गरज असते.
**पायडॅंटिक मॉडेल्स** आणि **टूल फंक्शन** यांना जोडून, आपण:

- एजंटच्या आउटपुटसाठी एक अचूक योजना (schema) परिभाषित करू शकतो
- प्रत्युत्तरांची आपोआप पडताळणी करू शकतो
- एजंटचे निकाल अॅप्लिकेशन लॉजिकमध्ये विश्वासार्हपणे एकत्र करू शकतो

आम्ही अशी टूलही सादर करतो ज्यामुळे एजंटच्या शिफारशी खऱ्या डेटावर आधारित असतात, कारण ती गंतव्यस्थान तपशील प्रदान करते.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## पॅटर्न 3: सिंगल रिस्पॉन्सिबिलिटी एजंट्स

संपूर्ण कामाला वेगवेगळ्या लक्ष केंद्रीत एजंट्समध्ये विभागण्यामुळे जटिल कामे सुलभ होतात, ज्यापैकी प्रत्येक एजंटची एकच जबाबदारी असते:

- **गंतव्य तज्ञ** जो ठिकाणे आणि उपलब्धतेबद्दल माहिती असतो
- **लॉजिस्टिक्स नियोजक** जो फ्लाइट, हॉटेल आणि प्रवास योजनेची काळजी घेतो

हे सॉफ्टवेअर अभियांत्रिकीतील *विभागणीच्या तत्त्वा*शी सुसंगत आहे — प्रत्येक एजंट स्वतंत्रपणे परीक्षण, देखभाल, आणि सुधारणा करणे सुलभ होते.


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## सारांश

या धड्यात आपण प्रवास शिफारस करणाऱ्या परिस्थितीसाठी तीन एजंटिक डिझाइन नमुने लागू केले:

| नमुना | मुख्य कल्पना | फायदा |
|---|---|---|
| **स्पष्ट सूचना** | व्यक्तिमत्व, जबाबदाऱ्या, आणि बंधने आधीच ठरवा | सुसंगत, ब्रँडशी सुसंगत एजंट वर्तन |
| **संरचित आउटपुट** | उत्तर स्वरूप म्हणून Pydantic मॉडेल वापरा | मान्यताप्राप्त, मशीन-योग्य निकाल |
| **एकल जबाबदारी** | प्रत्येक एजंटला एक लक्ष केंद्रीत काम द्या | सहज चाचणी, देखभाल, आणि संयोजन शक्य |

हे नमुने नैसर्गिकरीत्या एकत्र होतात — तुम्ही स्पष्ट सूचना आणि संरचित आउटपुट एकल जबाबदारी असलेल्या एजंटमध्ये एकत्र करून मजबूत, उत्पादनासाठी तयार प्रणाली तयार करू शकता.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**इस्तीफा**:  
हा दस्तऐवज AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. आम्ही अचूकतेसाठी प्रयत्न करत असलो तरी, कृपया लक्षात घ्या की स्वयंचलित अनुवादांमध्ये चुका किंवा अचूकतेचा अभाव असू शकतो. मूळ दस्तऐवज त्याच्या स्थानिक भाषेत अधिकृत स्रोत मानला जावा. महत्त्वाच्या माहितीसाठी व्यावसायिक मानवी अनुवाद शिफारसीय आहे. या अनुवादाच्या वापरामुळे उद्भवलेल्या कोणत्याही गैरसमजुती किंवा चुकीच्या अर्थाने आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
